# Getting set up

## Setting up the notebook

In [1]:
### Installing dependencies
!pip install openai

!apt-get update
!apt-get install -y iverilog

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
iverilog is already the newest version (11.0-1.1).
0 upgraded, 0 newly installed, 0 to remove 

In [2]:
def ask_deepseek(prompt):
    from openai import OpenAI
    client = OpenAI(
        api_key = "sk-4b4b50b8153147759b5530268e51a1f7",
        base_url = "https://api.deepseek.com"
    )
    completion = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=2048,
        temperature=0.7
    )
    return completion.choices[0].message.content

## Use LLM to generate Verilog code from natural language description

### Example 1: binary_to_bcd

In [3]:
! mkdir -p binary_to_bcd
! cd binary_to_bcd && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/binary_to_bcd_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1078  100  1078    0     0   8903      0 --:--:-- --:--:-- --:--:--  8983


In [4]:
verilog_generation_prompt = '''
I am trying to create a Verilog model binary_to_bcd_converter for a binary to binary-coded-decimal converter. It must meet the following specifications:
	- Inputs:
		- Binary input (5-bits)
	- Outputs:
		- BCD (8-bits: 4-bits for the 10's place and 4-bits for the 1's place)

How would I write a design that meets these specifications?
'''

In [5]:
ask_deepseek(verilog_generation_prompt)

'Here\'s a Verilog implementation for a 5-bit binary to BCD converter:\n\n## Method 1: Behavioral Implementation (Simple and Recommended)\n\n```verilog\nmodule binary_to_bcd_converter (\n    input [4:0] binary_in,    // 5-bit binary input (0-31)\n    output reg [7:0] bcd_out   // 8-bit BCD output [tens:ones]\n);\n\nalways @(*) begin\n    case (binary_in)\n        5\'d0:   bcd_out = 8\'h00;  // 00\n        5\'d1:   bcd_out = 8\'h01;  // 01\n        5\'d2:   bcd_out = 8\'h02;  // 02\n        5\'d3:   bcd_out = 8\'h03;  // 03\n        5\'d4:   bcd_out = 8\'h04;  // 04\n        5\'d5:   bcd_out = 8\'h05;  // 05\n        5\'d6:   bcd_out = 8\'h06;  // 06\n        5\'d7:   bcd_out = 8\'h07;  // 07\n        5\'d8:   bcd_out = 8\'h08;  // 08\n        5\'d9:   bcd_out = 8\'h09;  // 09\n        5\'d10:  bcd_out = 8\'h10;  // 10\n        5\'d11:  bcd_out = 8\'h11;  // 11\n        5\'d12:  bcd_out = 8\'h12;  // 12\n        5\'d13:  bcd_out = 8\'h13;  // 13\n        5\'d14:  bcd_out = 8\'h14;  // 1

Now you can use the openai library to generate text from your prompt.

Before using LLMs for Verilog generation, you have to first get the api-key:

openai llms: https://platform.openai.com/api-keys
deepseek llms: https://platform.deepseek.com/sign_in
free llms provided by NVIDIA: https://build.nvidia.com/explore/discover

from openai import OpenAI
import os
client = OpenAI(
    api_key = "sk-9dfe3571281d46c1a801a2c7592a726b"
)

completion = client.chat.completions.create(
  model = "gpt-4o-mini",
  messages=[{"role":"user","content":verilog_generation_prompt}],
  max_tokens=1024, # limits the maximum number of tokens (words or pieces of words) that the model will generate in its response.
  stream=False
)

print(completion.choices[0].message.content)

Next, extract the generated verilog code from LLM response.

In [6]:
output_verilog_code = '''
module binary_to_bcd_converter (
    input wire [4:0] binary_input, // 5-bit binary input
    output reg [7:0] bcd_output     // 8-bit BCD output (4 bits for tens, 4 bits for ones)
);

// Internal registers for BCD representation
reg [3:0] bcd_tens; // Tens place
reg [3:0] bcd_ones; // Ones place
integer i; // Loop variable

always @* begin
    // Initialize BCD values to 0
    bcd_tens = 4'b0000;
    bcd_ones = 4'b0000;

    // Convert binary to BCD using double-dabble algorithm
    for (i = 0; i < 5; i = i + 1) begin
        // Shift left the BCD digits to make room for the next binary bit
        if (bcd_tens >= 5)
            bcd_tens = bcd_tens + 4'b0011; // Add 3 to tens place if necessary

        if (bcd_ones >= 5)
            bcd_ones = bcd_ones + 4'b0011; // Add 3 to ones place if necessary

        // Shift BCD values left to add a new binary bit
        {bcd_tens, bcd_ones} = {bcd_tens, bcd_ones} << 1;

        // Add the next bit of the binary input
        bcd_ones[0] = binary_input[4 - i]; // Add the next bit of the binary input
    end

    // Combine BCD places into the final output
    bcd_output = {bcd_tens, bcd_ones};
end

endmodule
'''
filename = "binary_to_bcd/binary_to_bcd.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

Use iverilog to verify the correctness of LLM generated verilog

In [7]:
!cd binary_to_bcd/ && iverilog -g2012 -o binary_to_bcd.vvp binary_to_bcd.v binary_to_bcd_tb.v && vvp binary_to_bcd.vvp

Testing Binary-to-BCD Converter...
VCD info: dumpfile my_design.vcd opened for output.
All test cases passed!


### Example 2: Sequence detector

In [8]:
! mkdir -p sequence_detector
! cd sequence_detector && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/sequence_detector_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2114  100  2114    0     0  17404      0 --:--:-- --:--:-- --:--:-- 17471


In [9]:
verilog_generation_prompt = '''
I am trying to create a Verilog model for a sequence detector. It must meet the following specifications:
	- Inputs:
		- Clock
		- Active-low reset
		- Data (3 bits)
	- Outputs:
		- Sequence found

While enabled, it should detect the following sequence of binary input values:
	- 0b001
	- 0b101
	- 0b110
	- 0b000
	- 0b110
	- 0b110
	- 0b011
	- 0b101

How would I write a design that meets these specifications?
'''

In [10]:
ask_deepseek(verilog_generation_prompt)

'Here\'s a Verilog implementation for your sequence detector:\n\n```verilog\nmodule sequence_detector (\n    input wire clk,\n    input wire reset_n,      // Active-low reset\n    input wire [2:0] data,   // 3-bit input data\n    output reg sequence_found\n);\n\n// Define states for the FSM\ntypedef enum logic [3:0] {\n    IDLE,\n    S1,     // Expecting 0b001\n    S2,     // Expecting 0b101\n    S3,     // Expecting 0b110\n    S4,     // Expecting 0b000\n    S5,     // Expecting 0b110\n    S6,     // Expecting 0b110\n    S7,     // Expecting 0b011\n    S8      // Expecting 0b101 (final state)\n} state_t;\n\n// State registers\nstate_t current_state, next_state;\n\n// Sequence to detect\nlocalparam [2:0] SEQ1 = 3\'b001;\nlocalparam [2:0] SEQ2 = 3\'b101;\nlocalparam [2:0] SEQ3 = 3\'b110;\nlocalparam [2:0] SEQ4 = 3\'b000;\nlocalparam [2:0] SEQ5 = 3\'b110;\nlocalparam [2:0] SEQ6 = 3\'b110;\nlocalparam [2:0] SEQ7 = 3\'b011;\nlocalparam [2:0] SEQ8 = 3\'b101;\n\n// State transition logic (co

Next, extract the generated verilog code from LLM response.

In [11]:
output_verilog_code = '''
module sequence_detector (
    input         clk,
    input         reset_n,
    input  [2:0] data,
    output reg    sequence_found
);

    // State encoding
    typedef enum reg [3:0] {
        S_IDLE      = 4'b0000,
        S_001       = 4'b0001,
        S_101       = 4'b0010,
        S_110       = 4'b0011,
        S_000       = 4'b0100,
        S_011       = 4'b0101
    } state_t;

    // State variables
    state_t current_state, next_state;

    // Sequence found flag
    reg found_flag;

    // State transition
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            current_state <= S_IDLE;
        end else begin
            current_state <= next_state;
        end
    end

    // Next state logic
    always @(*) begin
        case (current_state)
            S_IDLE: begin
                if (data == 3'b001)
                    next_state = S_001;
                else
                    next_state = S_IDLE;
            end
            S_001: begin
                if (data == 3'b101)
                    next_state = S_101;
                else
                    next_state = S_IDLE;
            end
            S_101: begin
                if (data == 3'b110)
                    next_state = S_110;
                else
                    next_state = S_IDLE;
            end
            S_110: begin
                if (data == 3'b000)
                    next_state = S_000;
                else
                    next_state = S_IDLE;
            end
            S_000: begin
                if (data == 3'b110)
                    next_state = S_110;
                else
                    next_state = S_IDLE;
            end
            // Return to S_IDLE for any unrecognized states
            default: begin
                next_state = S_IDLE;
            end
        endcase
    end

    // Output logic
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            sequence_found <= 0;
        end else begin
            // Set found_flag only for successful transitions
            sequence_found <= (current_state == S_110 && next_state == S_011) ||
                              (current_state == S_011 && next_state == S_101);
        end
    end

endmodule
'''
filename = "sequence_detector/sequence_detector.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [12]:
!cd sequence_detector/ && iverilog -g2012 -o sequence_detector.vvp sequence_detector.v sequence_detector_tb.v && vvp sequence_detector.vvp

Error: Cycle 8, Expected: 1, Got: 0


### Example 3: Shift Register

In [13]:
! mkdir -p shift_register
! cd shift_register && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/shift_register_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1367  100  1367    0     0  12569      0 --:--:-- --:--:-- --:--:-- 12657


In [14]:
verilog_generation_prompt = '''
I am trying to create a Verilog model for a shift register. It must meet the following specifications:
	- Inputs:
		- Clock
		- Active-low reset
		- Data (1 bit)
		- Shift enable
	- Outputs:
		- Data (8 bits)

How would I write a design that meets these specifications?
'''

In [15]:
ask_deepseek(verilog_generation_prompt)

"Here's a Verilog model for an 8-bit shift register that meets your specifications:\n\n```verilog\nmodule shift_register (\n    input wire clk,          // Clock input\n    input wire rst_n,        // Active-low reset\n    input wire data_in,      // 1-bit data input\n    input wire shift_en,     // Shift enable\n    output reg [7:0] data_out // 8-bit data output\n);\n\n// Shift register implementation\nalways @(posedge clk or negedge rst_n) begin\n    if (!rst_n) begin\n        // Active-low reset: clear all bits\n        data_out <= 8'b0;\n    end\n    else if (shift_en) begin\n        // Shift right: MSB gets new data, others shift right\n        data_out <= {data_in, data_out[7:1]};\n    end\n    // If shift_en is low, data_out retains its value\nend\n\nendmodule\n```\n\n## Alternative Implementation (Left Shift)\n\nIf you need a left-shifting register instead (where LSB gets new data and bits shift left):\n\n```verilog\nmodule shift_register_left (\n    input wire clk,\n    input 

In [16]:
output_verilog_code = '''
module shift_register (
  input wire clk,        // Clock signal
  input wire reset_n,    // Active-low reset signal
  input wire data_in,    // Data input
  input wire shift_enable,
  output reg [7:0] data_out
);

always @(posedge clk or negedge reset_n) begin
  if (!reset_n) begin
    // Active-low reset, so when reset_n is 0, reset data_out to 0
    data_out <= 8'b00000000;
  end else if (shift_enable) begin
    // Shift data on the rising edge of the clock when shift_enable is 1
    data_out <= {data_out[6:0], data_in};
  end
end

endmodule
'''
filename = "shift_register/shift_register.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [17]:
!cd sequence_detector/ && iverilog -o sequence_detector.vvp sequence_detector.v sequence_detector_tb.v && vvp sequence_detector.vvp

sequence_detector.v:10: syntax error
sequence_detector.v:10: error: Invalid module instantiation
sequence_detector.v:20: syntax error
sequence_detector.v:20: error: Invalid module instantiation


### Example 4: Abro State Machine

In [18]:
! mkdir -p abro_state_machine
!cd abro_state_machine && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/abro_state_machine_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2762  100  2762    0     0  23116      0 --:--:-- --:--:-- --:--:-- 23210


In [19]:
verilog_generation_prompt = '''
I am trying to create a Verilog model for an ABRO state machine. It must meet the following specifications:
    - Inputs:
        - Clock
        - Active-low reset
        - A
        - B
    - Outputs:
        - O
        - State

Other than the main output from ABRO machine, it should output the current state of the machine for use in verification.

The states for this state machine should be one-hot encoded.

How would I write a design that meets these specifications?
'''

In [20]:
ask_deepseek(verilog_generation_prompt)

'Here\'s a Verilog implementation of an ABRO state machine with one-hot encoding:\n\n```verilog\nmodule abro_state_machine (\n    input wire clk,\n    input wire reset_n,      // Active-low reset\n    input wire A,\n    input wire B,\n    output reg O,\n    output reg [3:0] state  // One-hot encoded state\n);\n\n// One-hot encoded state definitions\nlocalparam [3:0] IDLE   = 4\'b0001;\nlocalparam [3:0] GOT_A  = 4\'b0010;\nlocalparam [3:0] GOT_B  = 4\'b0100;\nlocalparam [3:0] DONE   = 4\'b1000;\n\n// State register\nreg [3:0] next_state;\n\n// State transition logic (combinational)\nalways @* begin\n    // Default assignments\n    next_state = state;\n    O = 1\'b0;\n    \n    case (1\'b1)  // Synopsys full_case parallel_case\n        state[0]: begin // IDLE state\n            if (A && !B) begin\n                next_state = GOT_A;\n            end else if (!A && B) begin\n                next_state = GOT_B;\n            end else if (A && B) begin\n                next_state = DONE;\n  

In [21]:
output_verilog_code = '''
'''
filename = "abro_state_machine/abro_state_machine.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [22]:
!cd abro_state_machine/ && iverilog -o abro_state_machine.vvp abro_state_machine.v abro_state_machine_tb.v && vvp abro_state_machine.vvp

abro_state_machine_tb.v:12: error: Unknown module type: abro_state_machine
2 error(s) during elaboration.
*** These modules were missing:
        abro_state_machine referenced 1 times.
***


### Example 5: Dice Roller

In [23]:
!mkdir -p dice_roller
!cd dice_roller && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/dice_roller_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  3416  100  3416    0     0  23076      0 --:--:-- --:--:-- --:--:-- 23238


In [24]:
verilog_generation_prompt = '''
I am trying to create a Verilog model for a simulated dice roller. It must meet the following specifications:
    - Inputs:
        - Clock
        - Active-low reset
        - Die select (2-bits)
        - Roll
    - Outputs:
        - Rolled number (up to 8-bits)

The design should simulate rolling either a 4-sided, 6-sided, 8-sided, or 20-sided die, based on the input die select. It should roll when the roll input goes high and output the random number based on the number of sides of the selected die.

How would I write a design that meets these specifications?
'''

In [25]:
ask_deepseek(verilog_generation_prompt)

'Here\'s a complete Verilog implementation for your simulated dice roller:\n\n```verilog\nmodule dice_roller (\n    input wire clk,\n    input wire reset_n,      // Active-low reset\n    input wire [1:0] die_select,  // 2-bit die select\n    input wire roll,         // Trigger roll when high\n    output reg [7:0] rolled_number  // 8-bit output\n);\n\n// Define die types\nlocalparam D4  = 2\'b00;  // 4-sided die\nlocalparam D6  = 2\'b01;  // 6-sided die\nlocalparam D8  = 2\'b10;  // 8-sided die\nlocalparam D20 = 2\'b11;  // 20-sided die\n\n// Internal signals\nreg [31:0] lfsr;           // 32-bit LFSR for pseudo-random number generation\nreg roll_prev;             // Previous value of roll for edge detection\nwire roll_edge;            // Positive edge of roll signal\nreg [7:0] max_value;       // Maximum value for selected die\n\n// Edge detection for roll signal\nalways @(posedge clk or negedge reset_n) begin\n    if (!reset_n) begin\n        roll_prev <= 1\'b0;\n    end else begin\n 

In [26]:
output_verilog_code = '''
'''
filename = "dice_roller/dice_roller.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [27]:
!cd dice_roller/ && iverilog -o dice_roller.vvp dice_roller.v dice_roller_tb.v && vvp dice_roller.vvp

dice_roller_tb.v:10: error: Unknown module type: dice_roller
2 error(s) during elaboration.
*** These modules were missing:
        dice_roller referenced 1 times.
***


### Example 6: LFSR

In [28]:
!mkdir -p lfsr
!cd lfsr && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/lfsr_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1809  100  1809    0     0  19399      0 --:--:-- --:--:-- --:--:-- 19451


In [29]:
verilog_generation_prompt = '''
I am trying to create a Verilog model for an LFSR. It must meet the following specifications:
	- Inputs:
		- Clock
        - Active-low reset
	- Outputs:
		- Data (8-bits)

The initial state should be 10001010, and the taps should be at locations 1, 4, 6, and 7.

How would I write a design that meets these specifications?
'''

In [31]:
ask_deepseek(verilog_generation_prompt)

'Here\'s a Verilog implementation of an 8-bit LFSR with your specifications:\n\n```verilog\nmodule lfsr (\n    input wire clk,\n    input wire reset_n,      // Active-low reset\n    output reg [7:0] data    // 8-bit output\n);\n\n// Taps at positions 1, 4, 6, and 7 (using 0-indexing)\n// For 8-bit LFSR, we need to XOR bits 7, 4, 2, and 1\n// (since position 1 = bit[1], position 4 = bit[4], etc.)\nwire feedback;\n\n// Calculate feedback bit: XOR of bits 7, 4, 2, and 1\nassign feedback = data[7] ^ data[4] ^ data[2] ^ data[1];\n\nalways @(posedge clk or negedge reset_n) begin\n    if (!reset_n) begin\n        // Active-low reset: initialize to 10001010\n        data <= 8\'b10001010;\n    end else begin\n        // Shift left and insert feedback at LSB\n        data <= {data[6:0], feedback};\n    end\nend\n\nendmodule\n```\n\n## Alternative Implementation with Parameterized Taps:\n\n```verilog\nmodule lfsr (\n    input wire clk,\n    input wire reset_n,\n    output reg [7:0] data\n);\n\n//

In [30]:
output_verilog_code = '''
'''
filename = "lfsr/lfsr.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [32]:
!cd lfsr/ && iverilog -o lfsr.vvp lfsr.v lfsr_tb.v && vvp lfsr.vvp

lfsr_tb.v:9: error: Unknown module type: lfsr
2 error(s) during elaboration.
*** These modules were missing:
        lfsr referenced 1 times.
***


### Example 7: Sequence Generator

In [33]:
!mkdir -p sequence_generator
!cd sequence_generator && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/sequence_generator_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2344  100  2344    0     0  23399      0 --:--:-- --:--:-- --:--:-- 23440


In [34]:
verilog_generation_prompt = '''
I am trying to create a Verilog model for a sequence generator. It must meet the following specifications:
	- Inputs:
		- Clock
		- Active-low reset
		- Enable
	- Outputs:
		- Data (8 bits)

While enabled, it should generate an output sequence of the following hexadecimal values and then repeat:
	- 0xAF
	- 0xBC
	- 0xE2
	- 0x78
	- 0xFF
	- 0xE2
	- 0x0B
	- 0x8D

How would I write a design that meets these specifications?
'''

In [35]:
ask_deepseek(verilog_generation_prompt)

'Here\'s a Verilog implementation for your sequence generator:\n\n```verilog\nmodule sequence_generator (\n    input wire clk,\n    input wire reset_n,    // Active-low reset\n    input wire enable,\n    output reg [7:0] data\n);\n\n    // Define the sequence as a parameter array\n    parameter [7:0] SEQUENCE [0:7] = {\n        8\'hAF, 8\'hBC, 8\'hE2, 8\'h78, \n        8\'hFF, 8\'hE2, 8\'h0B, 8\'h8D\n    };\n    \n    // Internal counter to track position in sequence\n    reg [2:0] seq_counter;\n    \n    always @(posedge clk or negedge reset_n) begin\n        if (!reset_n) begin\n            // Active-low reset: reset counter and output\n            seq_counter <= 3\'b0;\n            data <= 8\'h00;\n        end\n        else if (enable) begin\n            // When enabled, output current sequence value\n            data <= SEQUENCE[seq_counter];\n            \n            // Increment counter, wrap around after last element\n            if (seq_counter == 3\'d7) begin\n               

In [36]:
output_verilog_code = '''
'''
filename = "sequence_generator/sequence_generator.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [37]:
!cd sequence_generator/ && iverilog -o sequence_generator.vvp sequence_generator.v sequence_generator_tb.v && vvp sequence_generator.vvp

sequence_generator_tb.v:10: error: Unknown module type: sequence_generator
2 error(s) during elaboration.
*** These modules were missing:
        sequence_generator referenced 1 times.
***


### Example 8: Traffic Light

In [38]:
!mkdir -p traffic_light/
!cd traffic_light/ && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/traffic_light_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2018  100  2018    0     0  12984      0 --:--:-- --:--:-- --:--:-- 13019


In [39]:
verilog_generation_prompt = '''
I am trying to create a Verilog model for a traffic light state machine. It must meet the following specifications:
    - Inputs:
        - Clock
        - Active-low reset
        - Enable
    - Outputs:
        - Red
        - Yellow
        - Green

The state machine should reset to a red light, change from red to green after 32 clock cycles, change from green to yellow after 20 clock cycles, and then change from yellow to red after 7 clock cycles.

How would I write a design that meets these specifications?
'''

In [40]:
ask_deepseek(verilog_generation_prompt)

'Here\'s a Verilog implementation of the traffic light state machine that meets your specifications:\n\n```verilog\nmodule traffic_light_fsm (\n    input wire clk,\n    input wire reset_n,    // Active-low reset\n    input wire enable,     // Enable signal\n    output reg red,\n    output reg yellow,\n    output reg green\n);\n\n// State definitions\nlocalparam [1:0] RED_STATE    = 2\'b00;\nlocalparam [1:0] GREEN_STATE  = 2\'b01;\nlocalparam [1:0] YELLOW_STATE = 2\'b10;\n\n// Timing constants\nlocalparam RED_TIME    = 32;    // 32 clock cycles\nlocalparam GREEN_TIME  = 20;    // 20 clock cycles\nlocalparam YELLOW_TIME = 7;     // 7 clock cycles\n\n// Internal registers\nreg [1:0] current_state, next_state;\nreg [5:0] counter;  // 6-bit counter (max 63 cycles)\n\n// State register (sequential logic)\nalways @(posedge clk or negedge reset_n) begin\n    if (!reset_n) begin\n        current_state <= RED_STATE;\n        counter <= 0;\n    end else if (enable) begin\n        current_state <=

In [41]:
output_verilog_code = '''
'''
filename = "traffic_light/traffic_light.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [42]:
!cd traffic_light/ && iverilog -o traffic_light.vvp traffic_light.v traffic_light_tb.v && vvp traffic_light.vvp

traffic_light_tb.v:32: syntax error
